# Loss-Grid Computer: Functional Eval Platform Suite

Runs the functional-evaluation redesign benchmark on Colab and saves a complete run bundle to Google Drive.

This notebook automates:
- Drive mount and persistent artifact storage
- repo loading from Drive or existing Colab workspace
- asset linking from Drive into the repo
- API probe capture via `src.functional_eval.api_pipeline`
- benchmark execution via `experiments.functional_eval_experiments`
- Drive export of probe output, benchmark summary, and run manifest

Target runtime is any CUDA GPU. The suite records hardware metadata and lets `RUN_LABEL` distinguish machines in filenames.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Root folder in Drive where assets live and benchmark bundles will be saved.
# Expected layout:
#   DRIVE_ROOT/
#     assets/
#       cifar-10-batches-py/
#       cifar10-resnet20-0.pkl
#     functional_eval_runs/
#     loss-grid-computer/
DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'
DRIVE_RESULTS_ROOT = f'{DRIVE_ROOT}/functional_eval_runs'

## 2. Load repo and install dependencies

In [ ]:
import os
import subprocess
from pathlib import Path

if 'DRIVE_ROOT' not in globals():
    DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
    DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'
    DRIVE_RESULTS_ROOT = f'{DRIVE_ROOT}/functional_eval_runs'

REPO_DIR = Path('/content/loss-grid-computer')
DRIVE_REPO_DIR = Path(DRIVE_ROOT) / 'loss-grid-computer'

if REPO_DIR.exists():
    print(f'Using existing workspace repo: {REPO_DIR}')
elif DRIVE_REPO_DIR.exists():
    REPO_DIR.symlink_to(DRIVE_REPO_DIR, target_is_directory=True)
    print(f'Linked repo from Drive: {REPO_DIR} -> {DRIVE_REPO_DIR}')
else:
    subprocess.check_call([
        'git',
        'clone',
        'https://github.com/hotz99/loss-grid-computer',
        str(REPO_DIR),
    ])
    print(f'Cloned repo: {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'torch',
    'torchvision',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'sympy>=1.13,<1.14',
])
print('Dependencies installed.')

## 3. Link assets from Drive

In [ ]:
import os
from pathlib import Path

assets_dst = Path('assets')
assets_src = Path(DRIVE_ASSETS_ROOT)

if assets_dst.exists() and not assets_dst.is_symlink():
    print(f'Assets already present in repo: {assets_dst.resolve()}')
elif assets_dst.is_symlink():
    print(f'Assets symlink already present: {assets_dst} -> {os.readlink(assets_dst)}')
elif assets_src.exists():
    assets_dst.symlink_to(assets_src)
    print(f'Linked assets from Drive: {assets_dst} -> {assets_src}')
else:
    assets_dst.mkdir(exist_ok=True)
    print(f'WARNING: {assets_src} not found. Create it in Drive and add the required files.')

required = [
    'assets/cifar-10-batches-py',
    'assets/cifar10-resnet20-0.pkl',
]
missing = [path for path in required if not Path(path).exists()]
if missing:
    raise FileNotFoundError('Missing required assets:\n' + '\n'.join(missing))

print('Required assets are present.')

## 4. Verify CUDA runtime

In [ ]:
import os
import torch

assert torch.cuda.is_available(), 'No CUDA GPU found. Change Runtime > Change runtime type > GPU.'
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu_name}')
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
print(f'CPU cores: {os.cpu_count()}')
print(f'Torch: {torch.__version__}')

## 5. Suite configuration

The default scenarios focus on the only near-winning candidate left: `functional_sequential`. The older vmap sweep and the full test-set run are opt-in because repeated runs show vmapped chunks are stable but slower than the warmed original baseline on the hardware measured so far.

In [ ]:
SEED = 1337
DEVICE = 'cuda'
GRID_RESOLUTION = 8
GRID_SCALE = 1.0
GPU_BATCH_SIZE = 32
MAX_MEMORY_FRACTION = 0.85
RUN_LABEL = gpu_name.lower().replace(' ', '-')
RUN_VMAP_REPRODUCTION = False
RUN_FULL_TEST_SET = False

PLATFORM_SCENARIOS = [
    {
        'name': 'functional_seq_1024_stability',
        'sample_count': 1024,
        'repeats': 7,
        'point_chunk_sizes': (),
    },
    {
        'name': 'functional_seq_2k_stability',
        'sample_count': 2048,
        'repeats': 5,
        'point_chunk_sizes': (),
    },
]

if RUN_VMAP_REPRODUCTION:
    PLATFORM_SCENARIOS.insert(0, {
        'name': 'prd_vmap_reproduction',
        'sample_count': 1024,
        'repeats': 3,
        'point_chunk_sizes': (1, 2, 4, 8, 16, 32, 64),
    })

if RUN_FULL_TEST_SET:
    PLATFORM_SCENARIOS.append({
        'name': 'functional_seq_full_test_set_stability',
        'sample_count': 0,  # 0 means the full CIFAR-10 test set in this repo's dataset wrapper.
        'repeats': 3,
        'point_chunk_sizes': (),
    })

print({
    'seed': SEED,
    'device': DEVICE,
    'grid_resolution': GRID_RESOLUTION,
    'grid_scale': GRID_SCALE,
    'gpu_batch_size': GPU_BATCH_SIZE,
    'max_memory_fraction': MAX_MEMORY_FRACTION,
    'run_label': RUN_LABEL,
    'run_vmap_reproduction': RUN_VMAP_REPRODUCTION,
    'run_full_test_set': RUN_FULL_TEST_SET,
    'scenarios': PLATFORM_SCENARIOS,
})

## 6. Run API probe and benchmark suite

In [ ]:
import importlib
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

repo_candidates = [Path.cwd(), Path.cwd().parent, REPO_DIR]
repo_root = next((path.resolve() for path in repo_candidates if (path / 'src').is_dir()), None)
if repo_root is None:
    checked = ', '.join(str(path.resolve()) for path in repo_candidates)
    raise ModuleNotFoundError(f"Could not locate repo root containing src/. Checked: {checked}")

functional_eval_dir = repo_root / 'src' / 'functional_eval'
if not functional_eval_dir.is_dir():
    raise ModuleNotFoundError(
        "This checkout does not include src/functional_eval required by this notebook. "
        f"repo_root={repo_root}. "
        "Sync the repository to a revision containing src/functional_eval and rerun from the top."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

for mod in list(sys.modules):
    if mod.startswith('src.functional_eval') or mod in {'src', 'src.schemas', 'src.workloads'}:
        del sys.modules[mod]

src_module = importlib.import_module('src')
src_file = Path(src_module.__file__).resolve()
if not src_file.is_relative_to(repo_root):
    raise ModuleNotFoundError(f"Imported 'src' from {src_file}, expected under {repo_root}")

api_module = importlib.import_module('src.functional_eval.api_pipeline')
suite_module = importlib.import_module('experiments.functional_eval_experiments')

api_result = api_module.run_pipeline(DEVICE, seed=SEED)
print(json.dumps(api_result, indent=2, sort_keys=True))

scenarios = tuple(
    suite_module.PlatformSuiteScenario(
        name=item['name'],
        sample_count=item['sample_count'],
        repeats=item['repeats'],
        batch_size=GPU_BATCH_SIZE,
        resolution=GRID_RESOLUTION,
        scale=GRID_SCALE,
        point_chunk_sizes=item['point_chunk_sizes'],
        max_memory_fraction=MAX_MEMORY_FRACTION,
    )
    for item in PLATFORM_SCENARIOS
)
suite_summary = suite_module.run_platform_suite(
    scenarios=scenarios,
    device=DEVICE,
    seed=SEED,
    run_label=RUN_LABEL,
)

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = Path(DRIVE_RESULTS_ROOT) / timestamp
run_dir.mkdir(parents=True, exist_ok=True)
scenario_dir = run_dir / 'scenarios'
scenario_dir.mkdir(exist_ok=True)

api_path = run_dir / 'api-pipeline.json'
api_path.write_text(json.dumps(api_result, indent=2, sort_keys=True), encoding='utf-8')

local_suite_path = Path(suite_summary['output_path'])
drive_suite_path = run_dir / 'platform-suite-summary.json'
shutil.copy2(local_suite_path, drive_suite_path)

scenario_copies = []
for item in suite_summary['scenarios']:
    scenario_name = item['scenario']['name']
    local_summary_path = Path(item['summary_path'])
    drive_summary_path = scenario_dir / f'{scenario_name}-experiment-summary.json'
    shutil.copy2(local_summary_path, drive_summary_path)
    scenario_copies.append({
        'scenario': scenario_name,
        'local_summary_path': str(local_summary_path),
        'drive_summary_path': str(drive_summary_path),
    })

manifest = {
    'created_at': timestamp,
    'repo_root': str(repo_root),
    'drive_root': DRIVE_ROOT,
    'run_dir': str(run_dir),
    'api_pipeline_path': str(api_path),
    'local_platform_suite_summary_path': str(local_suite_path),
    'drive_platform_suite_summary_path': str(drive_suite_path),
    'scenario_summaries': scenario_copies,
    'config': {
        'seed': SEED,
        'device': DEVICE,
        'grid_resolution': GRID_RESOLUTION,
        'grid_scale': GRID_SCALE,
        'gpu_batch_size': GPU_BATCH_SIZE,
        'max_memory_fraction': MAX_MEMORY_FRACTION,
        'run_label': RUN_LABEL,
        'run_vmap_reproduction': RUN_VMAP_REPRODUCTION,
        'run_full_test_set': RUN_FULL_TEST_SET,
        'scenarios': [
            {**item, 'point_chunk_sizes': list(item['point_chunk_sizes'])}
            for item in PLATFORM_SCENARIOS
        ],
    },
    'suite_summary': suite_summary,
}
manifest_path = run_dir / 'run-manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')

print(f'Run bundle saved to {run_dir}')
print(f'API probe: {api_path}')
print(f'Platform suite summary: {drive_suite_path}')
print(f'Manifest: {manifest_path}')

## 7. Candidate summary by scenario

In [ ]:
import pandas as pd

rows = []
for item in suite_summary['scenarios']:
    scenario = item['scenario']
    for row in item['candidate_summary']:
        taxonomy = row.get('taxonomy') or {}
        rows.append({
            'scenario': scenario['name'],
            'sample_count': scenario['sample_count'],
            'components': ','.join(taxonomy.get('components', [])),
            'sections': ','.join(taxonomy.get('applies_to_sections', [])),
            **row,
        })

candidate_df = pd.DataFrame(rows)
candidate_df = candidate_df.sort_values(['scenario', 'all_validations_passed', 'mean_speedup_vs_baseline'], ascending=[True, False, False], na_position='last')
candidate_df

## 8. Acceptance view

This view reports whether each scenario clears the `>~5%` speedup threshold on the current platform using paired repeat-level speedups and numerical validation.

In [ ]:
acceptance_df = pd.DataFrame([
    {
        'scenario': item['scenario']['name'],
        'sample_count': item['scenario']['sample_count'],
        **item['acceptance'],
    }
    for item in suite_summary['scenarios']
])
acceptance_df